<a href="https://colab.research.google.com/github/arashshams/Insurance-Policy-RAG/blob/new_dev/notebooks/02_embeddings_and_indexing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Insurance Policy RAG - Notebook 02: Embeddings & Indexing

This is the **second stage** of the Insurance Policy RAG pipeline.

**Goal:** turn the page-tagged text chunks produced by Notebook 01 into vector embeddings and store them in a persistent Chroma vector database on Google Drive, ready for retrieval in Notebook 03 (Q&A).

**Steps in this notebook:**
1. Mount Drive and locate the project folder
2. Create the Gemini client (API key read from Colab Secrets)
3. Load the chunks saved by Notebook 01 (`artifacts/chunks.json`)
4. Embed the chunks in batches with retry/backoff (free-tier friendly)
5. Persist the embeddings to a **fixed** Chroma directory on Drive so the index is reused across sessions
6. Run a quick retrieval sanity check

> Prerequisite: add your `GEMINI_API_KEY` in Colab **Secrets** (the key icon in the left sidebar) and enable notebook access.

In [1]:
# Install dependencies for this notebook (safe to re-run)
# On a fresh Colab runtime these packages are not pre-installed.
!pip install -q openai chromadb tqdm numpy > /dev/null 2>&1
print("Dependencies installed.")

Dependencies installed.


In [2]:
# Standard library
import os
import json
import time
import random

# Numerical utilities
import numpy as np
from tqdm import tqdm

# Mount Google Drive so we can read chunks and persist the Chroma DB
from google.colab import drive
drive.mount('/content/drive')

# Project root on Drive (must match Notebook 01)
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/Insurance-Policy-RAG')

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CHROMA_DIR    = PROJECT_ROOT / "chroma"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Artifacts dir:", ARTIFACTS_DIR)
print("Chroma dir:", CHROMA_DIR)

Mounted at /content/drive
Project root: /content/drive/MyDrive/Insurance-Policy-RAG
Artifacts dir: /content/drive/MyDrive/Insurance-Policy-RAG/artifacts
Chroma dir: /content/drive/MyDrive/Insurance-Policy-RAG/chroma


## 1. Create the Gemini Client

We call Gemini through its **OpenAI-compatible** endpoint, so the same `client` object handles both embeddings and (later) chat. The API key is read from Colab **Secrets** rather than hard-coded, so it never lives in the notebook or the repo.

To set it: open the **key icon** in the left sidebar, add a secret named `GEMINI_API_KEY`, paste your key, and toggle **Notebook access** on.

In [3]:
# Set up the Gemini client (Colab-only; reads the key from Colab Secrets)
from google.colab import userdata   # Colab helper to read secrets
from openai import OpenAI            # OpenAI-compatible client for Gemini

# The secret name must match exactly what you set in Colab Secrets.
gemini_api_key = userdata.get("GEMINI_API_KEY")

if not gemini_api_key:
    raise RuntimeError(
        "GEMINI_API_KEY not found. Add it in Colab Secrets (key icon in the "
        "left sidebar), name it exactly GEMINI_API_KEY, and enable notebook access."
    )

# OpenAI-compatible client pointing at Google's Generative Language endpoint
client = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# Models used in this notebook
EMBED_MODEL = "gemini-embedding-001"

# Quick check that the key works and the embedding model responds
_probe = client.embeddings.create(model=EMBED_MODEL, input="probe")
print("Gemini client ready. Embedding dimension:", len(_probe.data[0].embedding))

Gemini client ready. Embedding dimension: 3072


## 2. Load the Chunks from Notebook 01

Notebook 01 saved the page-tagged chunks to `artifacts/chunks.json`. Each record has an `id`, `text`, `page`, and `source`. We load them here and split the fields into the parallel lists Chroma expects (`ids`, `documents`, `metadatas`).

In [4]:
# Load the chunks produced by Notebook 01
CHUNKS_PATH = ARTIFACTS_DIR / "chunks.json"

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(
        f"{CHUNKS_PATH} not found. Run Notebook 01 (document ingestion) first."
    )

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

# Split into the parallel lists Chroma expects
ids       = [r["id"] for r in records]
texts     = [r["text"] for r in records]
metadatas = [{"page": r.get("page", -1), "source": r.get("source", "")} for r in records]

print(f"Loaded {len(records)} chunks from {CHUNKS_PATH.name}")
print("Example id:", ids[0], "| page:", metadatas[0]["page"])
print("First 120 chars:", texts[0][:120])

Loaded 45 chunks from chunks.json
Example id: doc_0 | page: 2
First 120 chars: Loblaw Companies Limited 1

Loblaw Companies Limited


Plan Contract Number:   G0100180,  G0100181

Group Policy Number:


## 3. Open the Persistent Chroma Database

We store the vector index in a **fixed** directory on Drive (`Insurance-Policy-RAG/chroma`) using `hnsw:space=cosine`.

> **Why fixed?** The original prototype used a timestamped path (`rag_chroma_db_{time.time()}`), which created a brand-new empty database on every run and meant the index could never be reused. A fixed path lets Notebook 03 open the same database later, and lets us re-run this notebook without duplicating work.

In [5]:
import chromadb

# Fixed, persistent location on Drive (reused across sessions and notebooks)
PERSIST_DIR = str(CHROMA_DIR)
COLLECTION_NAME = "insurance_policy_cvdb"

os.environ["ANONYMIZED_TELEMETRY"] = "False"
chromadb_client = chromadb.PersistentClient(path=PERSIST_DIR)

# Create or open the collection (cosine distance for normalized-embedding similarity)
existing_names = [c.name for c in chromadb_client.list_collections()]
if COLLECTION_NAME in existing_names:
    collection = chromadb_client.get_collection(COLLECTION_NAME)
else:
    collection = chromadb_client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

print("Chroma persist dir:", PERSIST_DIR)
print("Collection:", COLLECTION_NAME, "| current count:", collection.count())

Chroma persist dir: /content/drive/MyDrive/Insurance-Policy-RAG/chroma
Collection: insurance_policy_cvdb | current count: 0


## 4. Embed the Chunks (Batched, with Backoff)

We send the chunk texts to Gemini's embedding model in small batches. The free tier is rate-limited, so each batch retries with **exponential backoff** on failure. To keep the notebook safely re-runnable, we skip chunks whose ids are already present in the collection.

In [6]:
# Skip chunks already indexed so this cell is safe to re-run
try:
    existing = set(collection.get()["ids"])
except Exception:
    existing = set()

pending = [(i, t, m) for i, t, m in zip(ids, texts, metadatas) if i not in existing]
print(f"{len(existing)} already indexed | {len(pending)} to embed")

def embed_batch(batch_texts, max_retries=6, initial_sleep=2.0):
    """Embed a batch with exponential backoff for free-tier rate limits."""
    sleep_time = initial_sleep
    for attempt in range(max_retries):
        try:
            resp = client.embeddings.create(model=EMBED_MODEL, input=batch_texts)
            return [r.embedding for r in resp.data]
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = sleep_time + random.uniform(0, 1)
            print(f"  batch failed ({type(e).__name__}); retrying in {wait:.1f}s...")
            time.sleep(wait)
            sleep_time *= 2  # exponential backoff

BATCH = 16
if pending:
    p_ids   = [x[0] for x in pending]
    p_texts = [x[1] for x in pending]
    p_meta  = [x[2] for x in pending]

    for start in tqdm(range(0, len(p_texts), BATCH), desc="Embedding batches"):
        b_ids   = p_ids[start:start + BATCH]
        b_texts = p_texts[start:start + BATCH]
        b_meta  = p_meta[start:start + BATCH]

        b_emb = embed_batch(b_texts)

        # Upsert this batch immediately so progress survives interruptions
        collection.add(ids=b_ids, documents=b_texts, metadatas=b_meta, embeddings=b_emb)
        time.sleep(0.5)  # gentle pacing for the free tier

print("Done. Collection now holds", collection.count(), "chunks.")

0 already indexed | 45 to embed


Embedding batches: 100%|██████████| 3/3 [00:05<00:00,  1.80s/it]

Done. Collection now holds 45 chunks.


## 5. Retrieval Sanity Check

A quick end-to-end test: embed a sample question, query the collection, and print the top matches with their page numbers. This confirms the index is populated and returns sensible, on-topic chunks. Notebook 03 will build the full Q&A layer on top of this same collection.

In [7]:
def embed_query(text):
    resp = client.embeddings.create(model=EMBED_MODEL, input=[text])
    return resp.data[0].embedding

def retrieve(query, k=4):
    q_emb = embed_query(query)
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    return list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

# Sample query (adjust to your policy's content)
sample_query = "What is the coverage for short term disability?"
print("Query:", sample_query, "\n")

for doc, meta, dist in retrieve(sample_query, k=4):
    preview = doc[:200].replace("\n", " ")
    print(f"[page {meta.get('page')}] (distance={dist:.4f})")
    print(preview, "...\n")

Query: What is the coverage for short term disability? 

[page 6] (distance=0.2838)
Benefit Summary  Loblaw Companies Limited 5 Weekly Income (Short Term Disability)  The Weekly Income (Short Term Disability) Benefit is insured under Manulife Financial’s Policy G0100179.  Benefit Amo ...

[page 28] (distance=0.3167)
Your Group Benefits  Loblaw Companies Limited 27 Definition of Totally Disabled  Totally Disabled means a restriction or lack of abili ty due to an illness or injury which prevents you from performing ...

[page 30] (distance=0.3285)
Your Group Benefits  Loblaw Companies Limited 29 Tax Status of Benefits  The tax position of any payments you receive under this benefit depends on whether you or your employer pays the cost of the be ...

[page 32] (distance=0.3377)
Your Group Benefits  Loblaw Companies Limited 31 The Benefit  Benefit Amount  66.67% of monthly earnings, to a maximum benefit of $2,000 Non-Evidence Limit  $2,000 Qualifying Period  26 week(s)  Bene ...

